# Using Pre-trained Large Language Models on-premise

Learning goal: Provide a simple demonstration of using LLMs on premise. 

## First example: Google's Gemma

A small large language model by Google which is designed to fit in a single CPU but stay competitive (for its size). 

We can use the same pattern as the computer vision models of previous notebooks to use this model using Huggingface's `pipeline` object. Always remember to check the "Use this model" dropdown as different models may beed slightly different syntax. 

*This model will download more than 2GB of data to your local machine. Huggingface stores your models in a specific folder on your machine called the HF cache. As you keep downloading models, you can maintain your HF cache clean by running `huggingface-cli delete-cache`. Install this command with*

    pip install -U "huggingface_hub[cli]"

In [1]:
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-1b-it")


/Users/joseantonio.rodriguez15/Library/CloudStorage/OneDrive-UniversitatRamónLlull/projects-ES-FR62M6XQ1J/teaching_prototyping/consulta/PDAI26/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Device set to use mps:0


Now we can call the LLM with the `predict` method. (Don't expect anything close to the quality of ChatGPT for a small model. However these are enough for simple questions or mainstream tasks, like summarizing or sentiment analysis).

In [4]:
pipe.predict("What is the capital of France?")

[{'generated_text': "What is the capital of France?\n\nParis.\n\nLet me know if you'd like to try another question!"}]

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it")

In [15]:
prompt = "What is the capital of France? Respond in a single word: \n"
inputs = tokenizer(prompt, return_tensors='pt')

output = model.generate(**inputs, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)


In [33]:
import torch

prompt = "What is the capital of France? Respond in a single word:\n"

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]

generated_tokens = []

for step in range(5):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)

    # logits for next token
    logits = outputs.logits[:, -1, :]

    # convert to probabilities
    probs = torch.softmax(logits, dim=-1)

    # most likely token
    next_token_id = torch.argmax(probs, dim=-1)
    token_id = next_token_id.item()
    token_prob = probs[0, token_id].item()

    token_str = tokenizer.decode([token_id])

    print(f"Step {step+1}: token='{token_str}'  id={token_id}  prob={token_prob:.6f}")

    generated_tokens.append(token_id)

    # append token to sequence
    next_token_id = next_token_id.unsqueeze(0)
    input_ids = torch.cat([input_ids, next_token_id], dim=1)

print("\nGenerated token ids:", generated_tokens)
print("Generated text:", tokenizer.decode(generated_tokens))

Step 1: token='**'  id=1018  prob=0.288574
Step 2: token='Paris'  id=50429  prob=0.999984
Step 3: token='**'  id=1018  prob=0.999089
Step 4: token='
'  id=107  prob=0.953388
Step 5: token='<end_of_turn>'  id=106  prob=0.999294

Generated token ids: [1018, 50429, 1018, 107, 106]
Generated text: **Paris**
<end_of_turn>
